In [ ]:
import json
import os

from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
AI_GATEWAY_TOKEN = os.getenv("AI_GATEWAY_TOKEN")
CLOUDFLARE_ACCOUNT_ID = os.getenv("CLOUDFLARE_ACCOUNT_ID")

# 공급자 직접 경로 대신 Gateway의 OpenAI 호환 라우팅 경로를 사용한다.
client = OpenAI(
    api_key=AI_GATEWAY_TOKEN,
    base_url=f"https://gateway.ai.cloudflare.com/v1/{CLOUDFLARE_ACCOUNT_ID}/ai-gateway-course/compat",
)

response = client.chat.completions.create(
    # model="gemini-3.8-flash-high",
    # model="gpt-6-luna",
    model="dynamic/mix",
    messages=[
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": "넌 무슨 모델이지"},
    ],
    # 이 값은 쿼리에 사용할 수 있다.
    extra_headers={
        "cf-aig-metadata": json.dumps({"user_id":"1234567890","is_paid":True, "plan": "Pro"}),
        # Gateway 타임아웃 단위는 밀리초: 1000 = 1초 (테스트할 때 활성화)
        # "cf-aig-request-timeout": "1000"
    }
)

print("응답 모델:", response.model)
print(response.choices[0].message.content)

응답 모델: gpt-6.1-terra
저는 OpenAI의 ChatGPT입니다. 이 대화에서는 제 정확한 모델명이나 버전은 확인할 수 없어요.


In [23]:
# 실패 직후 실행: 새 요청 없이 마지막 HTTP 오류를 확인한다.
import sys

last_error = getattr(sys, "last_exc", None) or getattr(sys, "last_value", None)
error_response = getattr(last_error, "response", None)
if error_response is not None:
    print("HTTP:", error_response.status_code)
    print("본문:", error_response.text.replace(AI_GATEWAY_TOKEN, "[REDACTED]"))
    for header in ("cf-aig-log-id", "cf-aig-event-id", "cf-aig-provider", "cf-aig-model", "cf-ray"):
        print(f"{header}:", error_response.headers.get(header))
else:
    print("확인할 HTTP 오류가 없습니다.")

HTTP: 402
본문: {"success":false,"result":[],"messages":[],"error":[{"code":2021,"message":"Insufficient unified billing credits. Please add additional credits on the AI Gateway Cloudflare dashboard to continue using provider models."}],"name":"AiGatewayError","httpCode":402,"internalCode":2021,"message":"Insufficient unified billing credits. Please add additional credits on the AI Gateway Cloudflare dashboard to continue using provider models.","description":"Insufficient unified billing credits. Please add additional credits on the AI Gateway Cloudflare dashboard to continue using provider models."}
cf-aig-log-id: 01a1bfa7-5469-76d4-adfb-86d87e476ae9
cf-aig-event-id: 01a1bfa7-5469-76d4-adfb-86d87e476ae9
cf-aig-provider: openai
cf-aig-model: gpt-6.1-terra
cf-ray: a57bd0270956740b-ICN
